# Class 2: Deciding on Potential Impact (Sensitivity)

## GIS Vulnerability & Risk Assessment Course

**What You'll Learn in This Class:**
- How to **score potential impact** — a measure of how badly a hazard could affect an asset
- Why **building exposure** matters more than just knowing a parcel is exposed
- Why your **selected community assets** are more important to prioritize in mitigation planning
- How to use **spatial joins** to connect buildings to parcels and identify which properties are truly at risk

**The Big Question We're Answering:**
*If a flood hits, which properties would suffer the MOST?*

---

## What is "Potential Impact" or "Sensitivity"?

Potential impact (also called **sensitivity**) answers: **"How badly could this hazard affect this asset?"**

It's not enough to know a parcel is in a flood zone. We need to understand:
1. **Is there even a building on the parcel?** (A vacant lot has less impact than a house)
2. **What type of asset is it?** (Your selected community assets affect people and critical functions directly)

### Sensitivity Scoring (Our 4-Level System)

In this class, we'll score sensitivity as follows:

| Score | Level | Criteria |
|-------|-------|----------|
| **3** | **High** | Parcel exposed + Building exposed + Your selected community asset type |
| **2** | **Medium** | Parcel exposed + Building exposed + NOT a community asset type |
| **1** | **Low** | Parcel exposed + NO building exposed |
| **0** | **Not Exposed** | Parcel is NOT in the hazard zone |

### Why This Matters to Communities

- **Your selected community assets (score 3):** These are the assets you identified in Class 0 as most important to your community. Flooding them = highest direct community impact.
- **Other developed properties (score 2):** Important but less directly aligned with your community priorities.
- **Vacant land with no building (score 1):** Least urgent—no immediate people or businesses affected.

---

## What You'll Do Today

We'll follow these steps:

1. **Load your community asset selections** from Class 0 configuration
2. **Mount Google Drive** and load your data
3. **Load spatial data** (parcels, buildings, flood zones) from a GeoPackage
4. **Filter to the 100-year flood zone** — the hazard area we're analyzing
5. **Find exposed buildings** — which buildings actually sit in the flood zone
6. **Link buildings back to parcels** — which parcels have exposed buildings
7. **Calculate potential impact** — assign each parcel a sensitivity score (0, 1, 2, or 3) based on community assets
8. **Visualize the results** — map colored by impact level
9. **Analyze the statistics** — how many parcels at each risk level
10. **Save your work** back to the GeoPackage

Let's start!


## Step 1: Install and Load Our Tools

We'll use Python libraries designed for geographic data. Think of them as **specialized Excel sheets for maps**.

- **GeoPandas:** Works with geographic data (coordinates, boundaries, layers)
- **Pandas:** Handles tables of data
- **NumPy:** Does mathematical calculations
- **Matplotlib:** Creates maps and charts
- **Folium:** Makes interactive web maps

Run this cell first (it takes ~30 seconds). It only needs to run once per session.


In [ ]:
# ============================================================
# Install and load our mapping and data tools
# (Run this cell every time you open this notebook)
# ============================================================
!pip install geopandas fiona shapely pyproj requests folium seaborn contextily --quiet

import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import folium
import os
import json
import warnings
warnings.filterwarnings('ignore')

print("✓ All tools loaded successfully!")

## Step 2: Connect to Your Google Drive

This cell connects the notebook to your Google Drive so we can read and save your data files.

**What will happen:**
1. A popup will appear asking for permission
2. Click **"Allow"** to let this notebook access your Google Drive
3. A code will appear — copy it back into the text box below
4. Press Enter

Your data is stored in Google Drive at:
`/content/drive/MyDrive/MSER_510_VULNERABILTY/vulnerability_risk_data.gpkg`

In [ ]:
# === ENVIRONMENT SETUP ===
import os

try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = '/content/drive/MyDrive/MSER_510_VULNERABILTY'
    print("Google Drive connected!")
except Exception:
    BASE_DIR = './'
    print("Running locally.")

DATA_DIR = os.path.join(BASE_DIR, 'data')
OUTPUT_DIR = os.path.join(BASE_DIR, 'outputs')

os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# GeoPackage file paths for chained loading
INPUT_GPKG = os.path.join(DATA_DIR, 'class_1_exposure.gpkg')  # Load from Class 1
OUTPUT_GPKG = os.path.join(DATA_DIR, 'class_2_impact.gpkg')  # Save to Class 2
CLASS0_GPKG = os.path.join(DATA_DIR, 'vulnerability_risk_data.gpkg')  # Fallback for base layers

print(f"  Base directory: {BASE_DIR}")
print(f"  Data directory: {DATA_DIR}")
print(f"  Output directory: {OUTPUT_DIR}")

## Step 3: Load Spatial Layers from the GeoPackage

A **GeoPackage** is like a geographic database file — it can contain multiple **layers** (like sheets in an Excel workbook).

We'll load:
- **Parcels:** Property boundaries in our study area
- **Buildings:** Building footprints (which buildings are where)
- **Flood zones:** Polygons showing areas at risk of flooding

Each layer will be loaded as a **GeoDataFrame** — think of it as a spreadsheet with an extra geography column (coordinates/shapes).

In [ ]:
# Load data from Class 1's GeoPackage
print("Loading spatial data...")
print("=" * 60)

# Load parcels from Class 1 (has exposure, is_asset_1, is_asset_2 fields)
try:
    parcels = gpd.read_file(INPUT_GPKG, layer='parcels')
    print(f"\n✓ Loaded {len(parcels):,} parcels from Class 1")
    print(f"  Columns: {list(parcels.columns)}")
    print(f"  CRS: {parcels.crs}")
except Exception as e:
    print(f"Error loading parcels from Class 1: {e}")
    parcels = None

# Load flood zones from Class 1 GeoPackage (or fall back to Class 0)
try:
    flood_zones = gpd.read_file(INPUT_GPKG, layer='flood_zones_100yr')
    print(f"\n✓ Loaded {len(flood_zones):,} flood zone features from Class 1")
except Exception:
    try:
        flood_zones = gpd.read_file(CLASS0_GPKG, layer='flood_zones')
        print(f"\n✓ Loaded {len(flood_zones):,} flood zone features from Class 0")
    except Exception as e:
        print(f"Error loading flood zones: {e}")
        flood_zones = None

# Load buildings from Class 0
try:
    buildings = gpd.read_file(CLASS0_GPKG, layer='buildings')
    print(f"\n✓ Loaded {len(buildings):,} buildings from Class 0")
except Exception as e:
    print(f"Note: No buildings layer found: {e}")
    buildings = None

In [ ]:
# ============================================================
# LOAD COMMUNITY ASSET SELECTIONS
# ============================================================
# Your community asset parusecode lists are stored in
# course_config.json, which was created in Class 0 and can be
# edited in Class 1. From this point forward (Classes 2-8),
# these values are read-only.
#
# If you need to change them, go back to Class 0 or Class 1.
# ============================================================

# Fallback defaults — only used if course_config.json does not exist
# (i.e., you haven't run Class 0 yet)
default_asset_1 = ['100', '101', '105', '120', '121', '170', '173', '411', '416']
default_asset_2 = ['340', '341', '365', '405', '414', '415', '417', '421', '423',
                   '425', '426', '430', '431', '432', '434', '435', '438', '440',
                   '444', '446', '447', '448', '450', '454', '455', '456', '462',
                   '464', '466', '468', '470', '471', '472', '476', '477', '478',
                   '480', '481', '483', '490', '492', '494', '495', '512', '541',
                   '543', '544', '551', '554']

config_path = os.path.join(DATA_DIR, 'course_config.json')
if os.path.exists(config_path):
    with open(config_path, 'r') as f:
        course_config = json.load(f)
    community_asset_1 = course_config.get('community_asset_1', default_asset_1)
    community_asset_2 = course_config.get('community_asset_2', default_asset_2)
    print(f"Community assets loaded from: {config_path}")
else:
    community_asset_1 = default_asset_1
    community_asset_2 = default_asset_2
    print(f"Config not found — using defaults.")
    print(f"  (Run Class 0 first to set your own community assets)")

# Ensure they're lists of strings
community_asset_1 = [str(c) for c in community_asset_1]
community_asset_2 = [str(c) for c in community_asset_2]
community_assets_all = community_asset_1 + community_asset_2

print(f"  Asset 1 codes: {community_asset_1}")
print(f"  Asset 2 codes: {community_asset_2}")
print(f"\nTo change these, go back to Class 0 or Class 1 and re-run.")

## Step 4: Examine the Data

Let's look at a few sample rows from each layer to understand what we're working with.

In [ ]:
# Show the first few parcels
print("=" * 80)
print("SAMPLE PARCELS (First 3 rows)")
print("=" * 80)
if parcels is not None:
    # Display key columns only (not geometry which is long)
    display_cols = [col for col in parcels.columns if col != 'geometry']
    print(parcels[display_cols].head(3))
    print(f"\nTotal rows: {len(parcels)}")
    if 'exposure' in parcels.columns:
        print(f"Exposure field found: {parcels['exposure'].value_counts().to_dict()}")
else:
    print("Parcels not loaded")

print("\n" + "=" * 80)
print("SAMPLE BUILDINGS (First 3 rows)")
print("=" * 80)
if buildings is not None:
    display_cols = [col for col in buildings.columns if col != 'geometry']
    print(buildings[display_cols].head(3))
    print(f"\nTotal rows: {len(buildings)}")
else:
    print("Buildings not loaded")

print("\n" + "=" * 80)
print("SAMPLE FLOOD ZONES (First 3 rows)")
print("=" * 80)
if flood_zones is not None:
    display_cols = [col for col in flood_zones.columns if col != 'geometry']
    print(flood_zones[display_cols].head(3))
    print(f"\nTotal rows: {len(flood_zones)}")
    if 'flood_zone_type' in flood_zones.columns:
        print(f"Flood zone types: {flood_zones['flood_zone_type'].unique()}")
    if 'frequency_label' in flood_zones.columns:
        print(f"Frequency labels: {flood_zones['frequency_label'].unique()}")
else:
    print("Flood zones not loaded")


## Step 5: Filter to the 100-Year Flood Zone

We only care about properties exposed to **100-year floods** (also called **1% annual chance floods**).

**Why 100-year?**
- It's the standard used in flood insurance and planning
- It represents a 1% chance of occurring in any given year
- It balances between risk and feasibility of protection

We'll create a smaller dataset with **only** the 100-year flood zone polygons.

In [ ]:
# Filter to 100-year flood zone
# Look for columns that might identify the flood frequency
print("Examining flood_zones columns for frequency information...")

if flood_zones is not None:
    print(f"Columns: {list(flood_zones.columns)}")

    # Find the 100-year flood zone
    # (Could be labeled as '100-year', '1% annual chance', etc.)
    flood_100yr = None

    # Strategy 1: Look for 'frequency_label' or similar column
    if 'frequency_label' in flood_zones.columns:
        print(f"Frequency labels found: {flood_zones['frequency_label'].unique()}")
        flood_100yr = flood_zones[flood_zones['frequency_label'].str.contains('100', case=False, na=False)]

    # Strategy 2: Look for 'flood_zone_type' with '1PCT' or 'A'
    elif 'flood_zone_type' in flood_zones.columns:
        print(f"Flood zone types found: {flood_zones['flood_zone_type'].unique()}")
        flood_100yr = flood_zones[flood_zones['flood_zone_type'].isin(['1PCT', 'A', '0.01'])]

    # Strategy 3: Use all flood zones if we can't distinguish
    else:
        print("No frequency column found. Using all flood zones as 100-year approximation.")
        flood_100yr = flood_zones.copy()

    if flood_100yr is not None and len(flood_100yr) > 0:
        print(f"\n✓ Found {len(flood_100yr)} features in 100-year flood zone")
        # Merge all 100-year polygons into one geometry for faster processing
        flood_100yr_merged = flood_100yr.unary_union
        print(f"✓ Merged into single flood zone polygon for analysis")
    else:
        print("⚠ No 100-year flood zone found. Using entire flood_zones layer.")
        flood_100yr_merged = flood_zones.unary_union
else:
    print("Flood zones not available")
    flood_100yr_merged = None


## Step 6: Find Buildings in the 100-Year Flood Zone

**Spatial Join Concept (Review):**

A **spatial join** answers the question: *"Which features from Layer A touch or overlap features in Layer B?"*

Think of it like this:
- Layer A = Buildings (points or rectangles showing where buildings are)
- Layer B = Flood zone (polygon showing the hazard area)
- Question = "Which buildings are inside the flood zone?"

We use GeoPandas function `gpd.sjoin()` to do this efficiently.

**Result:** A new dataset with ONLY buildings that intersect the flood zone.

In [ ]:
# Find buildings exposed to the flood zone using a spatial join

if buildings is not None and len(buildings) > 0:
    # Create a GeoDataFrame for the 100-year flood zone
    flood_100yr_gdf = gpd.GeoDataFrame(
        {'flood_zone': [1]},
        geometry=[flood_100yr_merged],
        crs=buildings.crs
    )

    print("Finding buildings exposed to 100-year flood zone...")
    print(f"Total buildings: {len(buildings):,}")
    print(f"Flood zone CRS: {flood_100yr_gdf.crs}")
    print(f"Buildings CRS: {buildings.crs}")

    # Spatial join: which buildings fall inside the flood zone?
    buildings_in_flood = gpd.sjoin(
        buildings,
        flood_100yr_gdf,
        how='inner',
        predicate='intersects'
    )

    print(f"\n✓ Found {len(buildings_in_flood):,} buildings in the flood zone")
    print(f"  ({len(buildings_in_flood):,} out of {len(buildings):,} total buildings are exposed)")

    # Create a 'building_exposed' flag in the full buildings layer
    buildings['building_exposed'] = 0
    buildings.loc[buildings.index.isin(buildings_in_flood.index), 'building_exposed'] = 1

    print(f"\nBuilding exposure summary:")
    print(buildings['building_exposed'].value_counts().to_dict())
else:
    buildings_in_flood = None
    print("No building footprints available.")

## Step 7: Link Buildings to Parcels

Now we need to know: **"Which parcels have buildings that are exposed to the flood?"**

We'll do another spatial join to find which parcel each building sits on.

**Process:**
1. Take our exposed buildings
2. Find which parcel each building is inside
3. Mark those parcels as having an exposed building

**Result:** The parcels layer will get a new column `building_exposed` (1 if it has an exposed building, 0 if not).

In [ ]:
# Link exposed buildings back to their parcels
# If building footprints are available, spatial join to find which parcel
# each exposed building sits on. If not, we skip this — Step 9 will fall

if buildings_in_flood is not None and len(buildings_in_flood) > 0:
    print("Linking exposed buildings back to their parcels...")

    exposed_buildings_only = buildings_in_flood.copy()
    # Drop join artifacts from previous spatial join so gpd.sjoin works cleanly
    exposed_buildings_only = exposed_buildings_only.drop(
        columns=['index_right', 'flood_zone'], errors='ignore'
    )
    print(f"Exposed buildings to process: {len(exposed_buildings_only):,}")

    # Spatial join: which parcel is each building on?
    buildings_to_parcels = gpd.sjoin(
        exposed_buildings_only,
        parcels[['geometry']],
        how='left',
        predicate='intersects'
    )

    print(f"Building-to-parcel join complete")

    # Mark parcels that have exposed buildings
    exposed_parcel_indices = buildings_to_parcels['index_right'].dropna().unique()

    print(f"Parcels with exposed buildings: {len(exposed_parcel_indices):,}")

    parcels['building_exposed'] = 0
    parcels.loc[exposed_parcel_indices, 'building_exposed'] = 1

    print(f"\nParcel building exposure summary:")
    print(parcels['building_exposed'].value_counts().to_dict())
else:
    print("No building footprints available — skipping building-to-parcel link.")


## Step 8: Calculate Potential Impact Score (Separately for Each Asset Group)

We score Asset 1 and Asset 2 parcels **independently**. Only parcels belonging to an asset group
receive scores for that group. "Other" parcels (not in either asset group) are not scored.

**The Logic (applied separately for each asset group):**
```
IF parcel is NOT in this asset group:
    potential_impact = 0  (Not scored for this group)
ELSE IF parcel is NOT exposed (exposure_a1/a2 == 0):
    potential_impact = 0  (Not at risk)
ELSE IF parcel is exposed BUT no building is there:
    potential_impact = 1  (Low impact)
ELSE IF parcel is exposed AND building is there:
    potential_impact = 2  (Medium impact)
ELSE IF parcel is exposed AND building is there AND IS in this asset group:
    potential_impact = 3  (High impact)
```

This produces two columns: `potential_impact_a1` and `potential_impact_a2`.

In [ ]:
print("Calculating potential impact scores (separately for Asset 1 and Asset 2)...")
print(f"\nAsset 1 codes: {community_asset_1}")
print(f"Asset 2 codes: {community_asset_2}")

# Ensure required columns exist
if 'exposure' not in parcels.columns:
    print("WARNING: 'exposure' column not found from Class 1!")
    parcels['exposure'] = 0
if 'building_exposed' not in parcels.columns:
    parcels['building_exposed'] = 0
    print("WARNING: No building_exposed field found — defaulting to 0")

# Ensure asset flags exist (from Class 1)
if 'is_asset_1' not in parcels.columns:
    ca1_codes = [str(c) for c in community_asset_1]
    parcels['is_asset_1'] = parcels['parusecode'].astype(str).isin(ca1_codes).astype(int)
if 'is_asset_2' not in parcels.columns:
    ca2_codes = [str(c) for c in community_asset_2]
    parcels['is_asset_2'] = parcels['parusecode'].astype(str).isin(ca2_codes).astype(int)

# Ensure exposure_a1/exposure_a2 exist (from Class 1)
if 'exposure_a1' not in parcels.columns:
    parcels['exposure_a1'] = parcels['exposure'].where(parcels['is_asset_1'] == 1, 0).astype(int)
if 'exposure_a2' not in parcels.columns:
    parcels['exposure_a2'] = parcels['exposure'].where(parcels['is_asset_2'] == 1, 0).astype(int)

def score_potential_impact(parcels, asset_flag_col, exposure_col, label):
    """Score potential impact for one asset group.
    Only parcels where asset_flag_col == 1 get scored (others get 0).
    """
    is_asset = parcels[asset_flag_col] == 1
    conditions = [
        ~is_asset,                                                                         # Not in this asset group
        is_asset & (parcels[exposure_col] == 0),                                           # In group but not exposed
        is_asset & (parcels[exposure_col] == 1) & (parcels['building_exposed'] == 0),      # Exposed, no building
        is_asset & (parcels[exposure_col] == 1) & (parcels['building_exposed'] == 1),      # Exposed + building
    ]
    choices = [0, 0, 1, 3]
    scores = np.select(conditions, choices, default=0)
    
    # Print summary
    total_in_group = is_asset.sum()
    scored = pd.Series(scores)
    print(f"\n{label}: {total_in_group:,} parcels")
    for s in [0, 1, 3]:
        labels = {0: 'Not exposed / not in group', 1: 'Exposed (no building)', 3: 'Exposed + building (asset)'}
        count = (scored[is_asset] == s).sum() if s > 0 else (scored[is_asset] == 0).sum()
        if s == 0:
            count = (is_asset & (parcels[exposure_col] == 0)).sum()
        print(f"  Score {s} — {labels.get(s, '?')}: {count:,}")
    return scores

# Score each asset group independently
parcels['potential_impact_a1'] = score_potential_impact(parcels, 'is_asset_1', 'exposure_a1', 'Asset 1')
parcels['potential_impact_a2'] = score_potential_impact(parcels, 'is_asset_2', 'exposure_a2', 'Asset 2')

print("\n" + "=" * 60)
print("Potential impact scores calculated for both asset groups!")

## Step 9: Visualize Results on a Map

We'll create a map showing each parcel colored by its potential impact score:
- **Red:** High impact (score 3) — homes, businesses, critical infrastructure in flood zone
- **Yellow:** Medium impact (score 2) — other buildings in flood zone
- **Green:** Low impact (score 1) — parcels with no buildings in flood zone
- **Light Gray:** Not exposed (score 0) — outside flood zone

This gives a quick visual understanding of which neighborhoods are most at risk.

In [ ]:
# Create maps showing potential impact for Asset 1 and Asset 2 separately
impact_colors = {
    0: '#757575',  # Dark gray - not exposed / not scored
    1: '#FFB74D',  # Light orange - low impact
    3: '#BF360C'   # Red - high impact (exposed + building + asset)
}

impact_labels = {
    0: 'Not Scored (0)',
    1: 'Exposed, No Building (1)',
    3: 'Exposed + Building (3)'
}

fig, axes = plt.subplots(1, 2, figsize=(20, 8), facecolor='#2b2b2b')

for ax, col, label in zip(axes, ['potential_impact_a1', 'potential_impact_a2'], ['Asset 1', 'Asset 2']):
    ax.set_facecolor('#2b2b2b')
    
    # Plot all parcels as background
    parcels.plot(ax=ax, color='#3a3a3a', edgecolor='#444444', linewidth=0.2, alpha=0.4)
    
    # Plot scored parcels by impact level
    for score in [1, 3]:
        subset = parcels[parcels[col] == score]
        if len(subset) > 0:
            subset.plot(
                ax=ax,
                color=impact_colors[score],
                edgecolor='black',
                linewidth=0.5,
                alpha=0.8,
                label=f'{impact_labels[score]} (n={len(subset)})'
            )
    
    # Add flood zone boundary
    if flood_100yr_merged is not None:
        gpd.GeoSeries([flood_100yr_merged]).plot(
            ax=ax, edgecolor='#2B5797', facecolor='none', linewidth=2, label='100-Year Flood Zone'
        )
    
    ax.set_title(f'Potential Impact: {label}', fontsize=14, fontweight='bold', color='white')
    ax.legend(loc='upper right', fontsize=9, facecolor='#3a3a3a', edgecolor='#555555', labelcolor='white')
    ax.set_axis_off()

plt.tight_layout()
plt.show()
print(f"Total parcels: {len(parcels):,}")

In [ ]:
import contextily as ctx
from matplotlib.patches import Patch
import matplotlib.gridspec as gridspec

# Reproject to Web Mercator for basemap tiles
parcels_wm = parcels.to_crs(epsg=3857)

fig, axes = plt.subplots(1, 2, figsize=(20, 12))

impact_colors = {1: '#FFB74D', 3: '#BF360C'}

for ax, col, label in zip(axes, ['potential_impact_a1', 'potential_impact_a2'], ['Asset 1', 'Asset 2']):
    # Layer 1: All parcels - no fill, thin grey borders
    parcels_wm.plot(ax=ax, facecolor='none', edgecolor='#888888', linewidth=0.3)
    
    # Layer 2: Potential impact scores
    for score in [1, 3]:
        subset = parcels_wm[parcels_wm[col] == score]
        if len(subset) > 0:
            subset.plot(ax=ax, facecolor=impact_colors[score], edgecolor='none', alpha=0.85)
    
    # Layer 3: Buildings
    try:
        buildings_layer = gpd.read_file(CLASS0_GPKG, layer='buildings')
        buildings_wm = buildings_layer.to_crs(epsg=3857)
        buildings_wm.plot(ax=ax, facecolor='#3D3D3D', edgecolor='#2a2a2a', linewidth=0.1, alpha=0.7)
    except Exception:
        pass
    
    # Layer 4: Flood zones
    try:
        flood_zones_full = gpd.read_file(CLASS0_GPKG, layer='flood_zones')
        flood_wm = flood_zones_full.to_crs(epsg=3857)
        flood_colors = {'Floodway': '#2B5797', '100-year': '#8FABBE', '500-year': '#B4D4E7'}
        for flood_type in ['500-year', '100-year', 'Floodway']:
            flood_subset = flood_wm[flood_wm['flood_category'] == flood_type]
            if len(flood_subset) > 0:
                flood_subset.plot(ax=ax, facecolor=flood_colors.get(flood_type, '#B4D4E7'),
                                edgecolor='none', alpha=0.5)
    except Exception:
        pass
    
    ctx.add_basemap(ax, source=ctx.providers.CartoDB.Positron, zoom='auto')
    ax.set_axis_off()
    ax.set_title(f'Potential Impact: {label}', fontsize=14, fontweight='bold', pad=10)

# Export
png_path = os.path.join(OUTPUT_DIR, 'potential_impact_map.png')
plt.savefig(png_path, dpi=150, bbox_inches='tight', facecolor='white', edgecolor='none')
plt.show()
print(f"Map exported to: {png_path}")

## Export Potential Impact Map to PNG

We'll create a high-resolution PNG map showing the potential impact assessment with all layers properly styled and ordered.

## Step 10: Analyze Statistics

Let's create summary tables showing:
1. How many parcels are at each impact level
2. Total parcel values by impact level (if available)
3. Total structure/building values by impact level (if available)

These numbers help planners prioritize mitigation efforts and understand community vulnerability.

In [ ]:
print("=" * 80)
print("SUMMARY STATISTICS: Potential Impact Assessment (by Asset Group)")
print("=" * 80)

impact_labels = {0: 'Not Exposed / Not Scored', 1: 'Low (1)', 2: 'Medium (2)', 3: 'High (3)'}

for col, label in [('potential_impact_a1', 'Asset 1'), ('potential_impact_a2', 'Asset 2')]:
    asset_flag = 'is_asset_1' if 'a1' in col else 'is_asset_2'
    in_group = parcels[parcels[asset_flag] == 1]
    print(f"\n{'─' * 40}")
    print(f"{label} ({len(in_group):,} parcels)")
    print(f"{'─' * 40}")
    
    for score in sorted(in_group[col].unique()):
        count = (in_group[col] == score).sum()
        pct = (count / len(in_group)) * 100
        print(f"  Score {score} — {impact_labels.get(score, '?')}: {count:,} ({pct:.1f}%)")
    
    # Value summary if available
    value_cols = [c for c in parcels.columns if c in ['parval', 'landval']]
    if 'parval' in value_cols:
        at_risk = in_group[in_group[col] > 0]
        print(f"  Total parcel value at risk: ${at_risk['parval'].sum():,.0f}")

print("\n" + "=" * 80)

## Step 11: Save Your Work

We save the separate potential impact scores (`potential_impact_a1`, `potential_impact_a2`) back to the GeoPackage.
Class 3 will load these and continue scoring each asset group independently.

In [ ]:
# Save results to Class 2's own GeoPackage
# This does NOT overwrite Class 0 or Class 1 data.

import fiona

print("Saving results...")
print(f"Target: {OUTPUT_GPKG}")

try:
    # Save parcels with potential_impact scores (creates new GeoPackage)
    parcels.to_file(OUTPUT_GPKG, layer='parcels', driver='GPKG')
    print(f"✓ Saved parcels with potential_impact scores ({len(parcels):,} features)")

    # NOTE: We do NOT save the local 'flood_zones' variable here because
    # it only contains 100-year + Floodway (filtered for this class's analysis).
    # The full flood_zones layer (with 500-year) will be carried forward
    # from CLASS0_GPKG by the fallback logic below.

    # Copy forward all layers from INPUT_GPKG (except parcels which we just saved)
    if os.path.exists(INPUT_GPKG):
        input_layers = fiona.listlayers(INPUT_GPKG)
        for layer_name in input_layers:
            if layer_name == 'parcels':
                continue  # Already saved updated version
            try:
                layer_data = gpd.read_file(INPUT_GPKG, layer=layer_name)
                layer_data.to_file(OUTPUT_GPKG, layer=layer_name, driver='GPKG', mode='a')
                print(f"✓ Copied layer from input: {layer_name} ({len(layer_data)} features)")
            except Exception as e:
                print(f"  Note: Could not copy layer '{layer_name}': {e}")

        # Copy non-spatial tables via sqlite3
        try:
            import sqlite3
            conn_in = sqlite3.connect(INPUT_GPKG)
            conn_out = sqlite3.connect(OUTPUT_GPKG)
            cursor = conn_in.cursor()
            cursor.execute("SELECT name FROM sqlite_master WHERE type='table'")
            all_tables = [row[0] for row in cursor.fetchall()]
            system_tables = ['gpkg_contents', 'gpkg_geometry_columns', 'gpkg_spatial_ref_sys',
                            'gpkg_ogr_contents', 'gpkg_tile_matrix', 'gpkg_tile_matrix_set',
                            'sqlite_sequence', 'gpkg_extensions', 'gpkg_metadata',
                            'gpkg_metadata_reference']
            for table in all_tables:
                if table in system_tables or table in input_layers or table.startswith('rtree_') or table.startswith('trigger_'):
                    continue
                try:
                    df = pd.read_sql(f'SELECT * FROM "{table}"', conn_in)
                    if len(df) > 0:
                        df.to_sql(table, conn_out, if_exists='replace', index=False)
                        print(f"✓ Copied non-spatial table: {table} ({len(df)} rows)")
                except Exception:
                    pass
            conn_in.close()
            conn_out.close()
        except Exception:
            pass

    # Ensure base layers from Class 0 are included (full flood_zones with 500-year, buildings, study_area)
    if os.path.exists(CLASS0_GPKG):
        try:
            output_layers = fiona.listlayers(OUTPUT_GPKG)
            class0_layers = fiona.listlayers(CLASS0_GPKG)
            for layer_name in class0_layers:
                if layer_name not in output_layers:
                    try:
                        layer_data = gpd.read_file(CLASS0_GPKG, layer=layer_name)
                        layer_data.to_file(OUTPUT_GPKG, layer=layer_name, driver='GPKG', mode='a')
                        print(f"✓ Added base layer from Class 0: {layer_name} ({len(layer_data)} features)")
                    except Exception as e:
                        print(f"  Note: Could not copy base layer '{layer_name}': {e}")
        except Exception as e:
            print(f"  Note: Could not check Class 0 layers: {e}")

    # Final summary
    final_layers = fiona.listlayers(OUTPUT_GPKG)
    print(f"\n✓ All data saved to: {OUTPUT_GPKG}")
    print(f"  File size: {os.path.getsize(OUTPUT_GPKG) / 1024 / 1024:.1f} MB")
    print(f"  Layers: {final_layers}")
    print(f"\n  Class 3 will load from this file.")

except Exception as e:
    print(f"Error saving: {e}")

## Congratulations!

You've completed **Class 2: Deciding on Potential Impact (Sensitivity)**.

**What You Accomplished:**
- Loaded spatial data and community asset selections
- Used spatial joins to find exposed buildings
- Linked buildings back to their parcels
- Calculated **separate** potential impact scores for Asset 1 and Asset 2
- Each asset group was scored independently (`potential_impact_a1`, `potential_impact_a2`)
- Visualized and analyzed results
- Saved results for future classes

**Next Steps (Class 3):**
In the next class, we'll assess **Adaptive Capacity** — how well can each asset group recover from flooding?
Again, Asset 1 and Asset 2 will be scored separately.

## Appendix: Additional Resources and Concepts

### More About Sensitivity Analysis

**Sensitivity** and **Potential Impact** are the same thing — a measure of how vulnerable an asset is to a particular hazard.

The vulnerability assessment formula is:
```
RISK = HAZARD × SENSITIVITY × EXPOSURE
```

Breaking this down:
- **HAZARD:** How likely is the flood? (probability, return period)
- **SENSITIVITY:** How badly would it affect the asset? (the scores we just calculated)
- **EXPOSURE:** Is the asset in the hazard zone? (yes/no or degree of overlap)

### Why Asset Type Matters

Different communities care about different things:
- **Residential:** People's homes, families, mental health impacts
- **Commercial:** Jobs, local economy, tax base
- **Industrial:** Regional economy, supply chains
- **Government/Institutional:** Public services, emergency response
- **Agricultural:** Food production, rural livelihoods

By scoring these differently, we're saying: *"Loss of a house affects people directly; loss of a warehouse affects the economy."*

### Spatial Join Technical Details

When we do `gpd.sjoin(buildings, flood_zones, how='inner', predicate='intersects')`:
- `buildings` = left layer (data we want to keep)
- `flood_zones` = right layer (what we're checking against)
- `how='inner'` = only keep buildings that match
- `predicate='intersects'` = buildings that touch OR overlap the flood zone

Other spatial predicates:
- `'contains'` → left layer fully contains right layer
- `'within'` → left layer fully inside right layer
- `'touches'` → left and right share a boundary
- `'crosses'` → left and right cross each other
- `'overlaps'` → left and right overlap but neither fully contains the other

For exposure analysis, we usually use `'intersects'` because we care if ANY part of the building is in the hazard zone.

### Common Gotchas When Working with Spatial Data

1. **Coordinate System Mismatch:** All layers must be in the same CRS (Coordinate Reference System). Use `.to_crs()` to reproject if needed.
2. **Null Geometries:** Some rows might have invalid or missing geometry. Use `.is_valid` to check.
3. **Duplicate Rows:** Spatial joins can create duplicates if one feature overlaps multiple features. Use `drop_duplicates()` if needed.
4. **Large File Size:** GeoPackages with millions of features can be slow. Consider filtering or simplifying geometries first.

### References for Further Learning

- **GeoPandas Documentation:** https://geopandas.org/
- **Flood Mapping Resources:** https://www.fema.gov/flood-maps
- **Vulnerability Assessment:** https://www.ipcc.ch/ (search for "vulnerability")
- **GIS Spatial Analysis:** Longley et al., "Geographic Information Systems and Science" (textbook)

---

**End of Class 2 Notebook**
